In [14]:
!pip install torch-geometric --quiet

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [15]:
import os
for f in os.listdir('/kaggle/input/datasets/malakksalahh/bitcoinotc-csv'):
    print(f)

soc-sign-bitcoinotc.csv


In [16]:

cols = ['source', 'target', 'rating', 'time']
df = pd.read_csv('/kaggle/input/datasets/malakksalahh/bitcoinotc-csv/soc-sign-bitcoinotc.csv', names=cols)

print(df.shape)
print(df.head())

# Relabel node IDs to a contiguous 0..N-1 range (required by PyG)
all_nodes = pd.concat([df['source'], df['target']]).unique()
node_map = {node: i for i, node in enumerate(all_nodes)}
df['source'] = df['source'].map(node_map)
df['target'] = df['target'].map(node_map)

num_nodes = len(all_nodes)
print(f"Nodes: {num_nodes}, Edges: {len(df)}")

(35592, 4)
   source  target  rating          time
0       6       2       4  1.289242e+09
1       6       5       2  1.289242e+09
2       1      15       1  1.289243e+09
3       4       3       7  1.289245e+09
4      13      16       8  1.289254e+09
Nodes: 5881, Edges: 35592


In [17]:
# Binary label: DISTRUST = 1 (minority, what we care about catching), TRUST = 0
df['label'] = (df['rating'] <= 0).astype(int)

edge_index = torch.tensor(df[['source', 'target']].values.T, dtype=torch.long)

out_deg = df.groupby('source').size().reindex(range(num_nodes), fill_value=0)
in_deg = df.groupby('target').size().reindex(range(num_nodes), fill_value=0)
avg_rating_given = df.groupby('source')['rating'].mean().reindex(range(num_nodes), fill_value=0)
avg_rating_received = df.groupby('target')['rating'].mean().reindex(range(num_nodes), fill_value=0)

node_features = torch.tensor(
    np.stack([out_deg, in_deg, avg_rating_given, avg_rating_received], axis=1),
    dtype=torch.float
)
node_features = (node_features - node_features.mean(dim=0)) / (node_features.std(dim=0) + 1e-8)

data = Data(x=node_features, edge_index=edge_index)
print(data)

Data(x=[5881, 4], edge_index=[2, 35592])


In [18]:
edge_labels = torch.tensor(df['label'].values, dtype=torch.float)

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=df['label']
)

train_edges = edge_index[:, train_idx]
test_edges = edge_index[:, test_idx]
train_labels = edge_labels[train_idx]
test_labels = edge_labels[test_idx]

In [19]:
model = GATTrustModel(in_channels=node_features.shape[1], hidden_channels=32, heads=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# Class weighting: upweight the positive class (now Distrust, the minority)
num_distrust = (train_labels == 1).sum().item()
num_trust = (train_labels == 0).sum().item()
pos_weight = torch.tensor([num_trust / num_distrust])
print(f"pos_weight (upweighting Distrust): {pos_weight.item():.3f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)
train_edges, test_edges = train_edges.to(device), test_edges.to(device)
train_labels, test_labels = train_labels.to(device), test_labels.to(device)
pos_weight = pos_weight.to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, train_edges)
    loss = criterion(out, train_labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_out = model(data.x, data.edge_index, test_edges)
            test_probs = torch.sigmoid(test_out).cpu().numpy()
            test_preds = (test_probs > 0.5).astype(int)
            auc = roc_auc_score(test_labels.cpu().numpy(), test_probs)
            acc = accuracy_score(test_labels.cpu().numpy(), test_preds)
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Test AUC: {auc:.4f} | Test Acc: {acc:.4f}")

pos_weight (upweighting Distrust): 8.991
Epoch   0 | Loss: 1.5885 | Test AUC: 0.4212 | Test Acc: 0.1002
Epoch  10 | Loss: 1.1252 | Test AUC: 0.8112 | Test Acc: 0.7063
Epoch  20 | Loss: 0.9866 | Test AUC: 0.8276 | Test Acc: 0.8234
Epoch  30 | Loss: 0.8932 | Test AUC: 0.8425 | Test Acc: 0.8224
Epoch  40 | Loss: 0.8657 | Test AUC: 0.8609 | Test Acc: 0.8160
Epoch  50 | Loss: 0.8263 | Test AUC: 0.8689 | Test Acc: 0.7952
Epoch  60 | Loss: 0.8033 | Test AUC: 0.8746 | Test Acc: 0.8418
Epoch  70 | Loss: 0.8089 | Test AUC: 0.8778 | Test Acc: 0.8607
Epoch  80 | Loss: 0.8026 | Test AUC: 0.8818 | Test Acc: 0.8517
Epoch  90 | Loss: 0.7984 | Test AUC: 0.8852 | Test Acc: 0.8553


In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)
train_edges, test_edges = train_edges.to(device), test_edges.to(device)
train_labels, test_labels = train_labels.to(device), test_labels.to(device)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, train_edges)
    loss = criterion(out, train_labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_out = model(data.x, data.edge_index, test_edges)
            test_probs = torch.sigmoid(test_out).cpu().numpy()
            test_preds = (test_probs > 0.5).astype(int)
            auc = roc_auc_score(test_labels.cpu().numpy(), test_probs)
            acc = accuracy_score(test_labels.cpu().numpy(), test_preds)
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Test AUC: {auc:.4f} | Test Acc: {acc:.4f}")

Epoch   0 | Loss: 0.7701 | Test AUC: 0.8886 | Test Acc: 0.8694
Epoch  10 | Loss: 0.7534 | Test AUC: 0.8938 | Test Acc: 0.8578
Epoch  20 | Loss: 0.7740 | Test AUC: 0.8959 | Test Acc: 0.8560
Epoch  30 | Loss: 0.7642 | Test AUC: 0.8981 | Test Acc: 0.8640
Epoch  40 | Loss: 0.7418 | Test AUC: 0.8995 | Test Acc: 0.8451
Epoch  50 | Loss: 0.7491 | Test AUC: 0.9018 | Test Acc: 0.8653
Epoch  60 | Loss: 0.7287 | Test AUC: 0.9047 | Test Acc: 0.8855
Epoch  70 | Loss: 0.7355 | Test AUC: 0.9049 | Test Acc: 0.8990
Epoch  80 | Loss: 0.7103 | Test AUC: 0.9068 | Test Acc: 0.8852
Epoch  90 | Loss: 0.7229 | Test AUC: 0.9094 | Test Acc: 0.8820


In [21]:
from sklearn.metrics import classification_report

model.eval()
with torch.no_grad():
    test_out = model(data.x, data.edge_index, test_edges)
    test_probs = torch.sigmoid(test_out).cpu().numpy()
    test_preds = (test_probs > 0.5).astype(int)

print("Final Test AUC:", roc_auc_score(test_labels.cpu().numpy(), test_probs))
print("\nClassification Report:")
print(classification_report(test_labels.cpu().numpy(), test_preds, target_names=['Trust', 'Distrust']))

Final Test AUC: 0.9105090380293019

Classification Report:
              precision    recall  f1-score   support

       Trust       0.97      0.89      0.93      6406
    Distrust       0.45      0.79      0.57       713

    accuracy                           0.88      7119
   macro avg       0.71      0.84      0.75      7119
weighted avg       0.92      0.88      0.90      7119

